In [8]:
import os
import json
import pandas as pd
from maomao.utils.constants import *
from maomao.parsing.metadata_utils import *
from maomao.parsing.integrated_dataset_utils import *

#### Hemolytic dataset integration and label-consistency analysis
- This notebook performs large-scale integration of hemolytic peptide annotations from multiple public databases and published datasets. Each input source is provided as a preprocessed CSV file containing peptide sequences and source-specific hemolysis labels.

- All sequences from the different sources are pooled and deduplicated to create a unified sequence set. A pivot table is then constructed in which each row corresponds to a unique peptide sequence and each column represents a data source, storing the corresponding label or a placeholder value when the sequence is not reported by that source.

- Quality control is applied at the sequence level by removing peptides containing non-canonical amino acids and by filtering sequences outside a predefined length range. Filtering statistics and length distributions are tracked for reproducibility and later reporting.

- Source-specific labels are mapped onto the pivot table using a standardized encoding scheme (positive, negative, unlabeled, unknown). The notebook then evaluates label consistency across sources by counting per-sequence label occurrences and computing the proportion of positive and negative evidence.

- Based on cross-source agreement, each sequence is classified into high-level categories, including exclusive positive, exclusive negative, unlabeled-only, or ambiguous. Sequences with conflicting annotations are further stratified according to the percentage of positive labels to provide a graded measure of confidence.

- Finally, the curated dataset is split into non-overlapping output subsets (positive, negative, unlabeled, and ambiguous), and comprehensive dataset metadata is generated. All curated datasets and metadata are exported in a structured format suitable for downstream machine learning, benchmarking, and reproducible analysis.

In [9]:
name_task = "toxic_effect_classification"
output_folder = "../../processed_data/integrating_and_cleaning_data/hemolytic"

# PATH_EXPORT are imported from peptide_toxicity_classifier.constants.
# Update them in constants.py according to the required input and export paths.

- Reading all sources with label

In [ ]:
df_Abdelbakyetal = pd.read_csv(f"{PATH_EXPORT}/{name_task}/Abdelbaky et al./processed_hemolytic_dataset.csv")
df_Abdelbakyetal = df_Abdelbakyetal.rename(columns={"label": "hemolytic"})

In [ ]:
df_Almotairietal = pd.read_csv(f"{PATH_EXPORT}/{name_task}/Almotairi et al./processed_hemolytic_dataset.csv")
df_Almotairietal = df_Almotairietal.rename(columns={"label": "hemolytic"})

In [12]:
df_AMPDB_hemolytic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/AMPDB v1/processed_hemolytic_dataset.csv")
df_AMPDB_hemolytic = df_AMPDB_hemolytic.rename(columns={"label": "hemolytic"})

In [13]:
df_AMPDeep = pd.read_csv(f"{PATH_EXPORT}/{name_task}/AMPDeep/processed_hemolytic_dataset.csv")
df_AMPDeep = df_AMPDeep.rename(columns={"label": "hemolytic"})

In [ ]:
df_Bhatnagaretal = pd.read_csv(f"{PATH_EXPORT}/{name_task}/Bhatnagar et al./processed_hemolytic_dataset.csv")
df_Bhatnagaretal = df_Bhatnagaretal.rename(columns={"label": "hemolytic"})

In [15]:
df_BIOPEP_UWM_hemolytic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/BIOPEP-UWM/processed_hemolytic_dataset.csv")
df_BIOPEP_UWM_hemolytic = df_BIOPEP_UWM_hemolytic.rename(columns={"label": "hemolytic"})

In [16]:
df_CICERON_hemolytic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/CICERON/processed_hemolytic_dataset.csv")
df_CICERON_hemolytic = df_CICERON_hemolytic.rename(columns={"label": "hemolytic"})

In [17]:
df_ConsAMPHemo_hemolytic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/ConsAMPHemo/processed_hemolytic_dataset.csv")
df_ConsAMPHemo_hemolytic = df_ConsAMPHemo_hemolytic.rename(columns={"label": "hemolytic"})

In [18]:
df_DRAMP_hemolytic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/DRAMP/processed_hemolytic_dataset.csv")
df_DRAMP_hemolytic = df_DRAMP_hemolytic.rename(columns={"label": "hemolytic"})

In [19]:
df_HAPPENN_hemolytic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/HAPPENN/processed_hemolytic_dataset.csv")
df_HAPPENN_hemolytic = df_HAPPENN_hemolytic.rename(columns={"label": "hemolytic"})

In [20]:
df_HemoDL_hemolytic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/HemoDL/processed_hemolytic_dataset.csv")
df_HemoDL_hemolytic = df_HemoDL_hemolytic.rename(columns={"label": "hemolytic"})

In [21]:
df_HemoFuse_hemolytic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/HemoFuse/processed_hemolytic_dataset.csv")
df_HemoFuse_hemolytic = df_HemoFuse_hemolytic.rename(columns={"label": "hemolytic"})

In [22]:
df_Hemolytik_hemolytic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/Hemolytik/processed_hemolytic_dataset.csv")
df_Hemolytik_hemolytic = df_Hemolytik_hemolytic.rename(columns={"label": "hemolytic"})

In [23]:
df_hemolytik_2_hemolytic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/hemolytik 2.0/processed_hemolytic_dataset.csv")
df_hemolytik_2_hemolytic = df_hemolytik_2_hemolytic.rename(columns={"label": "hemolytic"})

In [24]:
df_hemonet_hemolytic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/hemonet/processed_hemolytic_dataset.csv")
df_hemonet_hemolytic = df_hemonet_hemolytic.rename(columns={"label": "hemolytic"})

In [25]:
df_HemoPI_hemolytic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/HemoPI/processed_hemolytic_dataset.csv")
df_HemoPI_hemolytic = df_HemoPI_hemolytic.rename(columns={"label": "hemolytic"})

In [26]:
df_HemoPI2_0_hemolytic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/HemoPI2.0/processed_hemolytic_dataset.csv")
df_HemoPI2_0_hemolytic = df_HemoPI2_0_hemolytic.rename(columns={"label": "hemolytic"})

In [27]:
df_HEPAD_hemolytic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/HEPAD/processed_hemolytic_dataset.csv")
df_HEPAD_hemolytic = df_HEPAD_hemolytic.rename(columns={"label": "hemolytic"})

In [28]:
df_HLPpred_Fuse_hemolytic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/HLPpred-Fuse/processed_hemolytic_dataset.csv")
df_HLPpred_Fuse_hemolytic = df_HLPpred_Fuse_hemolytic.rename(columns={"label": "hemolytic"})

In [29]:
df_HMAMP_main_hemolytic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/HMAMP-main/processed_hemolytic_dataset.csv")
df_HMAMP_main_hemolytic = df_HMAMP_main_hemolytic.rename(columns={"label": "hemolytic"})

In [30]:
df_iAMPCN_hemolytic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/iAMPCN/processed_hemolytic_dataset.csv")
df_iAMPCN_hemolytic = df_iAMPCN_hemolytic.rename(columns={"label": "hemolytic"})

In [31]:
df_MultiPep_hemolytic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/MultiPep/processed_hemolytic_dataset.csv")
df_MultiPep_hemolytic = df_MultiPep_hemolytic.rename(columns={"label": "hemolytic"})

In [32]:
df_Multitox_hemolytic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/Multitox/processed_hemolytic_dataset.csv")
df_Multitox_hemolytic = df_Multitox_hemolytic.rename(columns={"label": "hemolytic"})

In [ ]:
df_Plisson_hemolytic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/Plisson et al./processed_hemolytic_dataset.csv")
df_Plisson_hemolytic = df_Plisson_hemolytic.rename(columns={"label": "hemolytic"})

In [34]:
df_QSVM_PEPTIDE_hemolytic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/QSVM-PEPTIDE/processed_hemolytic_dataset.csv")
df_QSVM_PEPTIDE_hemolytic = df_QSVM_PEPTIDE_hemolytic.rename(columns={"label": "hemolytic"})

In [ ]:
df_Zhao_et_al_hemolytic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/Zhao et al./processed_hemolytic_dataset.csv")
df_Zhao_et_al_hemolytic = df_Zhao_et_al_hemolytic.rename(columns={"label": "hemolytic"})

In [36]:
df_peptipedia_hemolytic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/Peptipedia2.0/processed_hemolytic_dataset.csv")
df_peptipedia_hemolytic = df_peptipedia_hemolytic.rename(columns={"label": "hemolytic"})

In [83]:
df_Hemolytik20_new = pd.read_csv(f"{PATH_EXPORT}/{name_task}/Hemolytik2.0_new/processed_hemolytik2.0_new_dataset.csv")
df_Hemolytik20_new = df_Hemolytik20_new.rename(columns={"label": "hemolytic"})

- Reading all sources with unlabel

In [38]:
df_hemolytic_pred_unlabel = pd.read_csv(f"{PATH_EXPORT}/{name_task}/Hemolytic-Pred/detected_unlabel_sequences.csv")
df_hemolytic_pred_unlabel = df_hemolytic_pred_unlabel.rename(columns={"label": "hemolytic"})

In [39]:
df_hemonet_unlabel = pd.read_csv(f"{PATH_EXPORT}/{name_task}/hemonet/detected_unlabel_sequences.csv")
df_hemonet_unlabel = df_hemonet_unlabel.rename(columns={"label": "hemolytic"})

In [ ]:
df_Karasevetal_unlabel = pd.read_csv(f"{PATH_EXPORT}/{name_task}/Karasev et al./detected_unlabel_sequences.csv")
df_Karasevetal_unlabel = df_Karasevetal_unlabel.rename(columns={"label": "hemolytic"})

In [ ]:
df_Plissonetal_unlabel = pd.read_csv(f"{PATH_EXPORT}/{name_task}/Plisson et al./detected_unlabel_sequences.csv")
df_Plissonetal_unlabel = df_Plissonetal_unlabel.rename(columns={"label": "hemolytic"})

- Collecting all sequences for activity

In [84]:
df_list_hemolytic = [
    df_Abdelbakyetal, df_Almotairietal, df_AMPDB_hemolytic, df_AMPDeep,
    df_Bhatnagaretal, df_BIOPEP_UWM_hemolytic, df_CICERON_hemolytic,
    df_ConsAMPHemo_hemolytic, df_DRAMP_hemolytic, df_HAPPENN_hemolytic,
    df_HemoDL_hemolytic, df_HemoFuse_hemolytic, df_Hemolytik_hemolytic,
    df_hemolytik_2_hemolytic, df_hemonet_hemolytic, df_HemoPI_hemolytic,
    df_HemoPI2_0_hemolytic, df_HEPAD_hemolytic, df_HLPpred_Fuse_hemolytic,
    df_HMAMP_main_hemolytic, df_Zhao_et_al_hemolytic, df_QSVM_PEPTIDE_hemolytic,
    df_Plisson_hemolytic, df_Multitox_hemolytic, df_peptipedia_hemolytic,
    df_iAMPCN_hemolytic, df_MultiPep_hemolytic, df_Hemolytik20_new,
    df_hemolytic_pred_unlabel, df_hemonet_unlabel, df_Karasevetal_unlabel, 
    df_Plissonetal_unlabel
]
unique_sequence_hemolytic = count_unique_sequence(df_list_hemolytic)

42225


- Create pivote dataset

In [85]:
df_pivote = create_pivote(unique_sequence_hemolytic)

- Removing sequences with non canonical residues 

In [86]:
n_before_canon = df_pivote.shape[0]
df_pivote["is_canon"] = df_pivote["sequence"].apply(check_sequence)
n_after_canon = df_pivote[df_pivote["is_canon"]].shape[0]

In [45]:
print(df_pivote["is_canon"].value_counts())
df_pivote = df_pivote[df_pivote["is_canon"]]

is_canon
True     39230
False     2955
Name: count, dtype: int64


- Filter sequences by length

In [87]:
df_pivote["length"] = df_pivote["sequence"].str.len()
df_pivote["length"].describe()

count    42225.000000
mean        44.378449
std        162.529556
min          1.000000
25%         14.000000
50%         21.000000
75%         33.000000
max      18141.000000
Name: length, dtype: float64

In [88]:
n_before_length = n_after_canon
df_pivote["filter_length"] = df_pivote["length"].apply(check_length)
n_after_length = df_pivote[df_pivote["filter_length"]].shape[0]

In [89]:
df_pivote["filter_length"].value_counts()

filter_length
True     38729
False     3496
Name: count, dtype: int64

In [90]:
length_series = df_pivote[df_pivote["filter_length"]]["length"]

length_dist = {
    "min": length_series.min(),
    "max": length_series.max(),
    "mean": length_series.mean(),
    "median": length_series.median()
}

In [91]:
df_pivote = df_pivote[df_pivote["filter_length"]]
df_pivote.shape

(38729, 4)

In [92]:
df_pivote = df_pivote.drop(columns=["is_canon", "filter_length", "length"])

In [93]:
df_list_hemolytic = [("Abdelbaky et al.", df_Abdelbakyetal), 
                    ("Almotairi et al.", df_Almotairietal),
                    ("AMPDB", df_AMPDB_hemolytic),
                    ("AMPDeep", df_AMPDeep),
                    ("Bhatnagar et al.", df_Bhatnagaretal),
                    ("BIOPEP-UWM", df_BIOPEP_UWM_hemolytic),
                    ("CICERON", df_CICERON_hemolytic),
                    ("ConsAMPHemo", df_ConsAMPHemo_hemolytic),
                    ("DRAMP", df_DRAMP_hemolytic),
                    ("HAPPENN", df_HAPPENN_hemolytic),
                    ("HemoDL", df_HemoDL_hemolytic),
                    ("HemoFuse", df_HemoFuse_hemolytic),
                    ("Hemolytik", df_Hemolytik_hemolytic),
                    ("hemolytik 2.0", df_hemolytik_2_hemolytic),
                    ("HemoNet", df_hemonet_hemolytic),
                    ("HemoPI", df_HemoPI_hemolytic),
                    ("HemoPI2.0", df_HemoPI2_0_hemolytic),
                    ("HEPAD", df_HEPAD_hemolytic),
                    ("HLPpred-Fuse", df_HLPpred_Fuse_hemolytic),
                    ("HMAMP-main", df_HMAMP_main_hemolytic),
                    ("Zhao et al.", df_Zhao_et_al_hemolytic),
                    ("QSVM-peptide", df_QSVM_PEPTIDE_hemolytic),
                    ("Plisson et al.", df_Plisson_hemolytic),
                    ("MultiTox", df_Multitox_hemolytic),
                    ("Peptipedia2.0", df_peptipedia_hemolytic),
                    ("iAMPCN", df_iAMPCN_hemolytic),
                    ("Multipep", df_MultiPep_hemolytic),
                    ("Hemolytik20_new", df_Hemolytik20_new),
                    ("Hemolytic-Pred unlabel", df_hemolytic_pred_unlabel),
                    ("HemoNet unlabel", df_hemonet_unlabel),
                    ("Karasev et al. unlabel", df_Karasevetal_unlabel),
                    ("Plisson et al. unlabel", df_Plissonetal_unlabel)
                ]

In [94]:
for source, dataset in df_list_hemolytic:
    dataset = dataset[["sequence", "hemolytic"]]
    dataset = dataset.drop_duplicates(subset="sequence")
    mapping = dataset.set_index("sequence")["hemolytic"]

    # Mapear sin explotar memoria
    df_pivote[source] = (
        df_pivote["sequence"]
        .map(mapping)
        .fillna(999)
        .astype("int16")
    )

In [95]:
df_pivote.head(5)

,sequence,Abdelbaky et al.,Almotairi et al.,AMPDB,AMPDeep,Bhatnagar et al.,BIOPEP-UWM,CICERON,ConsAMPHemo,DRAMP,...,Plisson et al.,MultiTox,Peptipedia2.0,iAMPCN,Multipep,Hemolytik20_new,Hemolytic-Pred unlabel,HemoNet unlabel,Karasev et al. unlabel,Plisson et al. unlabel
1,RDRLKDLGSEKIERLRGFQLSGQSDRIRRKILSEFGL,999,999,999,0,999,999,999,999,999,...,999,999,999,999,999,999,999,999,999,999
2,APQSITIATLAKNCICTTGVLCNLWILLKGGWYWIV,999,999,999,999,999,999,999,999,999,...,999,999,999,999,999,999,999,999,999,2
3,DDITKMLVATTH,999,999,999,0,999,999,999,999,999,...,999,999,999,999,999,999,999,999,999,999
4,LYANVLRLFKPTFPHLAGYSTFCLKAT,999,999,999,999,999,999,999,999,999,...,999,999,999,999,999,999,999,999,999,2
5,GSPIQCAETCFIGKCYTEELGCTCTAFLCMKN,999,999,999,999,999,999,999,999,999,...,999,999,999,0,999,999,999,999,999,999


- Working with pivote for detecting ambiguous sequences 

In [96]:
df_pivote = process_count_labels(df_pivote)

In [97]:
df_pivote["negative"].value_counts() # Includes sources labeled as nevative (0) and unlabeled (2)

negative
True     19757
False    18972
Name: count, dtype: int64

In [98]:
df_pivote["exclusive_0"].value_counts() # Includes only sources labeled as nevative (0) 

exclusive_0
False    19635
True     19094
Name: count, dtype: int64

In [99]:
df_pivote["positive"].value_counts() # Includes sources labeled as positive (1) and unlabeled (2)

positive
False    31711
True      7018
Name: count, dtype: int64

In [100]:
df_pivote["exclusive_1"].value_counts() # Includes only sources labeled as positive (1) 

exclusive_1
False    32442
True      6287
Name: count, dtype: int64

In [101]:
df_pivote["only_unlabel"].value_counts() # Includes only sources unlabeled (2)

only_unlabel
False    33678
True      5051
Name: count, dtype: int64

In [102]:
df_pivote.sort_values(by="percentage_1", ascending=False)

,sequence,Abdelbaky et al.,Almotairi et al.,AMPDB,AMPDeep,Bhatnagar et al.,BIOPEP-UWM,CICERON,ConsAMPHemo,DRAMP,...,counts_0,counts_unlabel,counts_unknown,positive,negative,exclusive_1,exclusive_0,only_unlabel,percentage_0,percentage_1
8,ATYrTGrATrESLSGVEISGrLYrLR,999,999,999,1,999,999,999,999,999,...,0,0,30,True,False,True,False,False,0.0,100.0
10,GKPRXYTPRPTSHPRPIRV,999,999,999,1,999,999,999,999,999,...,0,0,31,True,False,True,False,False,0.0,100.0
42213,GLLALISWIKRKRQQX,999,999,999,999,999,999,999,999,999,...,0,0,31,True,False,True,False,False,0.0,100.0
42217,QRKIRVRLSA,999,999,999,1,999,999,999,999,999,...,0,0,31,True,False,True,False,False,0.0,100.0
42219,GFGMAlKLLKKVL,999,999,999,999,999,999,999,999,999,...,0,0,31,True,False,True,False,False,0.0,100.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42222,RQIKIWFQNRRMKWKA,999,999,999,999,999,999,999,999,999,...,1,0,31,False,True,False,True,False,100.0,0.0
45,GLFFAINIRKALCELGIRYFKDVLNYLLA,999,999,999,999,999,999,999,999,999,...,0,1,31,False,False,False,False,True,0.0,0.0
47,QAFKTFTPDWNKIRNDAKRMQDNLEQMKKKFNLNL,999,999,999,999,999,999,999,999,999,...,1,0,31,False,True,False,True,False,100.0,0.0
42193,VNFPLTFCAKLCDYKGYSILIS,999,999,999,999,999,999,999,999,999,...,0,1,31,False,False,False,False,True,0.0,0.0


- Splitting data into negative, positive, and with amiguous data

In [103]:
negative = df_pivote[df_pivote["negative"]]

In [104]:
only_negative = df_pivote[df_pivote["exclusive_0"]]

In [105]:
positive = df_pivote[df_pivote["positive"]]

In [106]:
only_positive = df_pivote[df_pivote["exclusive_1"]]

In [107]:
only_unlabel = df_pivote[df_pivote["only_unlabel"]]

In [108]:
df_ambiguous = df_pivote[(df_pivote["positive"] == False) & (df_pivote["negative"] == False) & (df_pivote["only_unlabel"] == False)]

- Processing ambiguous data

In [109]:
df_ambiguous =  categorize_percentage(df_ambiguous)

In [110]:
df_ambiguous["Category_pbb"].value_counts()

Category_pbb
40-50    3271
10-20     760
70-80     583
80-90     493
20-30     485
30-40     445
>90       345
60-70     224
>0        157
50-60     140
Name: count, dtype: int64

- Working with metada

In [111]:
seq_stats = {
    "canonical": {
        "before": n_before_canon,
        "after": n_after_canon
    },
    "length": {
        "before": n_before_length,
        "after": n_after_length,
        "min": MIN_LENGTH_SEQUENCE,
        "max": MAX_LENGTH_SEQUENCE
    },
    "length_dist": length_dist
}

metadata = build_dataset_metadata(
    task="hemolytic",
    source_list=df_list_hemolytic,
    pivote_df=df_pivote,
    outputs={
        "only_positive": only_positive,
        "only_negative": only_negative,
        "ambiguous": df_ambiguous
    },
    seq_stats=seq_stats,
    filters={
        "canonical_residues": True,
        "length_filter": True
    }
)

metadata

{'task': 'hemolytic',
 'generated_at': '2026-09-03T12:38:42.904030',
 'sources': {'n_unique_sequences': {'Abdelbaky et al.': 5851,
   'Almotairi et al.': 5400,
   'AMPDB': 833,
   'AMPDeep': 16191,
   'Bhatnagar et al.': 756,
   'BIOPEP-UWM': 63,
   'CICERON': 63,
   'ConsAMPHemo': 5841,
   'DRAMP': 508,
   'HAPPENN': 3532,
   'HemoDL': 5437,
   'HemoFuse': 5437,
   'Hemolytik': 2471,
   'hemolytik 2.0': 4960,
   'HemoNet': 3273,
   'HemoPI': 2191,
   'HemoPI2.0': 1926,
   'HEPAD': 2996,
   'HLPpred-Fuse': 4365,
   'HMAMP-main': 1104,
   'Zhao et al.': 188,
   'QSVM-peptide': 2195,
   'Plisson et al.': 2351,
   'MultiTox': 260,
   'Peptipedia2.0': 1304,
   'iAMPCN': 22646,
   'Multipep': 463,
   'Hemolytik20_new': 2459,
   'Hemolytic-Pred unlabel': 1235,
   'HemoNet unlabel': 35,
   'Karasev et al. unlabel': 944,
   'Plisson et al. unlabel': 7274}},
 'filters': {'canonical_residues': {'applied': True},
  'length_filter': {'applied': True, 'min_length': 5, 'max_length': 70}},
 'sequence

- Exporting data

In [112]:
os.makedirs(output_folder, exist_ok=True)

In [113]:
with open(f"{output_folder}/metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

In [114]:
negative.shape

(19757, 44)

In [115]:
only_negative.shape

(19094, 44)

In [116]:
positive.shape

(7018, 44)

In [117]:
only_positive.shape

(6287, 44)

In [118]:
only_unlabel.shape

(5051, 44)

In [119]:
df_ambiguous.shape

(6903, 45)

In [120]:
negative.to_csv(f"{output_folder}/negative.csv", index=False)
only_negative.to_csv(f"{output_folder}/only_negative.csv", index=False)

In [121]:
positive.to_csv(f"{output_folder}/positive.csv", index=False)
only_positive.to_csv(f"{output_folder}/only_positive.csv", index=False)

In [122]:
only_unlabel.to_csv(f"{output_folder}/only_unlabel.csv", index=False)

In [123]:
df_ambiguous.to_csv(f"{output_folder}/ambiguous_data.csv", index=False)